# B&T vehicle-class split — passenger cars vs the rest (WP-A2, corrected)

**dtj7-qync** is WIDE: one row per `(plaza, collection_date)` with numeric toll-class columns
`class_1 … class_39` and a `total`. MTA defines toll **Class 1 = Cars** (2-axle passenger
vehicles, ~90% of crossings), so we trust only `class_1` (cars) and the validated `total`, and
take non-passenger = `total − class_1` (robust to whatever classes 2–39 mean and to any overlap
in the class_31–39 range).

**Run order:** Cell 1 pulls a sample and prints the per-class 2019 shares + a partition check —
**confirm `class_1` is the dominant (~88%) stream** (i.e. it really is cars) before trusting the
split. Cells 2–3 pull the full table, aggregate, and write the preview.


In [ ]:
# Cell 1 — pull sample, CONFIRM class_1 is the dominant (cars) stream
import os, sys, pandas as pd, requests, io
sys.path.insert(0,'.'); import bt_vehicle_class as btvc
APP_TOKEN = os.environ.get('SOCRATA_APP_TOKEN')
url=f'https://{btvc.SOCRATA_DOMAIN}/resource/{btvc.DATASET_ID}.csv'
hdr={'X-App-Token':APP_TOKEN} if APP_TOKEN else {}
# a 2019 slice is enough to see the class shares
samp=pd.read_csv(io.StringIO(requests.get(url,headers=hdr,
     params={'$where':"collection_date between '2019-01-01' and '2019-12-31'",'$limit':50000},timeout=90).text))
print('class columns:', btvc.class_columns(samp))
print('\npartition (sum of classes vs total):', btvc.partition_check(samp))
print('\n2019 class shares (top 8):'); print(btvc.class_shares(samp,2019).head(8).to_string(index=False))
print('\n-> confirm class_1 share ~0.85-0.90 (cars). If a different class dominates, tell me before Cell 3.')


In [ ]:
# Cell 2 — full pull + monthly split
raw=btvc.pull_dtj7(app_token=APP_TOKEN)
print('pulled rows:', len(raw))
print('partition (full):', btvc.partition_check(raw))
monthly=btvc.to_monthly(raw)   # passenger=class_1 (cars); other=total-class_1
print('months:', monthly.index.min().date(),'->',monthly.index.max().date(),'| n=',len(monthly))
print(monthly.tail(6).round(0))


In [ ]:
# Cell 3 — recovery + preview export
print('Recovery (trailing-12mo / base-year mean):'); print(btvc.recovery(monthly).to_string(index=False))
print('\nAnnual means:'); print(btvc.annual(monthly).round(0).to_string(index=False))
prev=monthly.reset_index()[['period_start','passenger','other','total']]
prev.columns=['period_start','passenger_cars','other_classes','total']
prev.insert(1,'service','Bridges and Tunnels'); prev.insert(2,'measure_type','vehicle_traffic_by_class')
prev.insert(3,'data_basis','observed_bt_daily')
prev.to_csv('bt_vehicle_class_monthly_preview.csv',index=False)
r=btvc.recovery(monthly).set_index('group')
print('\nwrote bt_vehicle_class_monthly_preview.csv rows:',len(prev))
print('HEADLINE: cars(class_1) recov vs 2019 = {:.2f}; other = {:.2f}. Send me this CSV + the Cell-1 shares.'.format(
      r.loc['passenger','recov_2019'], r.loc['other','recov_2019']))
